# Belajar YOLOv8 bertahap: deteksi safety helmet di pabrik

Notebook ini membawa kamu dari nol sampai punya model yang bisa menandai **orang yang tidak memakai helm**.
Setiap tahap punya penjelasan singkat, kode yang bisa langsung dijalankan, dan pertanyaan kecil untuk dicek sendiri.

| Tahap | Yang dipelajari |
|---|---|
| 1 | Menyiapkan lingkungan (Colab + GPU) |
| 2 | Mengambil dataset Hard Hat Workers dari Kaggle |
| 3 | Mengenali data: gambar, anotasi, distribusi kelas |
| 4 | Mengubah anotasi Pascal VOC (XML) ke format YOLO |
| 5 | Membagi data train / validation |
| 6 | Mengecek ulang label (sanity check) |
| 7 | Training YOLOv8 |
| 8 | Membaca kurva training |
| 9 | Evaluasi: precision, recall, mAP, confusion matrix, heatmap |
| 10 | Inference: menghitung orang tanpa helm |
| 11 | Menyimpan hasil untuk website |

**Cara menjalankan di Google Colab:** `Runtime → Change runtime type → T4 GPU`, lalu jalankan sel dari atas ke bawah.
Di laptop tanpa GPU tetap bisa, tapi isi `MAX_IMAGES` dengan angka kecil (misalnya 300) supaya training selesai dalam hitungan menit, bukan jam.

## Tahap 0 — Gambaran besar

**Object detection** = menjawab dua pertanyaan sekaligus untuk setiap objek di gambar: *apa* objeknya (kelas) dan *di mana* letaknya (bounding box).

**YOLO** (*You Only Look Once*) melakukan keduanya dalam satu kali lewat jaringan saraf, sehingga cukup cepat untuk CCTV real-time.
Arsitekturnya terdiri dari tiga bagian:

- **Backbone**: mengekstrak fitur visual (tepi, tekstur, bentuk helm).
- **Neck**: menggabungkan fitur dari beberapa skala, supaya kepala yang jauh (kecil) maupun dekat (besar) sama-sama terdeteksi.
- **Head**: memprediksi kotak dan kelas untuk setiap lokasi.

Kita tidak melatih dari nol. Kita pakai **transfer learning**: mulai dari `yolov8n.pt` yang sudah belajar dari dataset COCO (80 jenis objek), lalu disesuaikan untuk 2 kelas:

| id | kelas | arti di pabrik |
|---|---|---|
| 0 | `helmet` | kepala memakai helm → aman |
| 1 | `no_helmet` | kepala tanpa helm → **pelanggaran K3** |

## Tahap 1 — Menyiapkan lingkungan

In [ ]:
%pip install -q ultralytics kagglehub

In [ ]:
import json, random, shutil, xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("GPU tersedia:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Semua angka yang boleh kamu ubah dikumpulkan di satu sel di bawah ini.

- `EPOCHS`: berapa kali model melihat seluruh data train. 30 cukup untuk hasil yang layak.
- `IMGSZ`: ukuran sisi gambar saat training. Gambar dataset berukuran 416 px; 640 membuat kepala kecil lebih jelas tapi training lebih lambat.
- `BATCH`: jumlah gambar per langkah. Turunkan kalau muncul error *out of memory*.
- `MAX_IMAGES`: `None` = pakai semua 5000 gambar. Isi angka kecil untuk percobaan cepat.

In [ ]:
EPOCHS = 30
IMGSZ = 640
BATCH = 16
MAX_IMAGES = None      # contoh: 300 untuk percobaan cepat di CPU
VAL_RATIO = 0.2
SEED = 42
BASE_MODEL = "yolov8n.pt"   # n = nano (paling ringan). Coba juga yolov8s.pt

WORK = Path.cwd()
DATASET = WORK / "dataset"      # hasil konversi format YOLO
RUNS = WORK / "runs"            # hasil training -> runs/helmet
CLASSES = {"helmet": 0, "head": 1}   # 'head' di dataset = kepala tanpa helm
NAMES = ["helmet", "no_helmet"]

## Tahap 2 — Mengambil dataset

Dataset: [Hard Hat Workers (Kaggle)](https://www.kaggle.com/datasets/andrewmvd/hard-hat-detection), 5000 gambar dengan anotasi Pascal VOC.
`kagglehub` bisa mengunduh dataset publik tanpa login. Kalau muncul error autentikasi, jalankan `kagglehub.login()` dan masukkan API token dari halaman akun Kaggle-mu.

Kalau dataset sudah ada di komputer, isi `DATA_SRC` dengan folder yang berisi `images/` dan `annotations/`, dan download akan dilewati.

In [ ]:
DATA_SRC = WORK / "archive"
if not (DATA_SRC / "annotations").exists():
    import kagglehub
    DATA_SRC = Path(kagglehub.dataset_download("andrewmvd/hard-hat-detection"))

IMG_SRC, ANN_SRC = DATA_SRC / "images", DATA_SRC / "annotations"
xml_files = sorted(ANN_SRC.glob("*.xml"))
print("Folder dataset:", DATA_SRC)
print("Jumlah gambar  :", len(list(IMG_SRC.iterdir())))
print("Jumlah anotasi :", len(xml_files))

## Tahap 3 — Mengenali data

Sebelum training, selalu lihat datanya dulu. Satu file anotasi Pascal VOC berisi ukuran gambar dan daftar objek.
Setiap objek punya `name` (kelas) dan `bndbox` dalam **piksel**: `xmin, ymin, xmax, ymax`.

In [ ]:
print(xml_files[0].read_text()[:700])

In [ ]:
def read_voc(xml_path):
    # Baca satu file VOC -> (nama_gambar, lebar, tinggi, [(kelas, xmin, ymin, xmax, ymax), ...])
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    w, h = int(float(size.find("width").text)), int(float(size.find("height").text))
    objects = []
    for obj in root.findall("object"):
        b = obj.find("bndbox")
        box = [float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax")]
        objects.append((obj.find("name").text, *box))
    return root.find("filename").text, w, h, objects

counts = Counter(o[0] for x in xml_files for o in read_voc(x)[3])
print(counts)
plt.figure(figsize=(5, 3))
plt.bar(counts.keys(), counts.values(), color=["#2b8a54", "#c8102e", "#888"][: len(counts)])
plt.title("Jumlah objek per kelas"); plt.ylabel("objek"); plt.show()

**Amati:** kelas `helmet` jauh lebih banyak daripada `head`. Ini disebut **ketidakseimbangan kelas** (*class imbalance*).
Model cenderung lebih jago mengenali kelas mayoritas, jadi nanti perhatikan *recall* kelas `no_helmet`.
Label `person` sangat sedikit, jadi kita abaikan.

In [ ]:
def show_voc(xml_path):
    fname, w, h, objects = read_voc(xml_path)
    img = Image.open(IMG_SRC / fname).convert("RGB")
    d = ImageDraw.Draw(img)
    for name, x1, y1, x2, y2 in objects:
        color = {"helmet": "lime", "head": "red"}.get(name, "yellow")
        d.rectangle([x1, y1, x2, y2], outline=color, width=2)
        d.text((x1, y1 - 10), name, fill=color)
    return img

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, x in zip(axes, random.Random(1).sample(xml_files, 3)):
    ax.imshow(show_voc(x)); ax.set_title(x.stem); ax.axis("off")
plt.show()

## Tahap 4 — Mengubah anotasi VOC ke format YOLO

YOLO membaca satu file `.txt` per gambar. Setiap baris = satu objek:

```
kelas  x_tengah  y_tengah  lebar  tinggi
```

Semua angka **dinormalisasi** ke 0–1 (dibagi lebar/tinggi gambar), jadi label tetap benar walaupun gambar di-resize.

$$x_{tengah} = \frac{x_{min}+x_{max}}{2\,W}, \quad y_{tengah} = \frac{y_{min}+y_{max}}{2\,H}, \quad lebar = \frac{x_{max}-x_{min}}{W}, \quad tinggi = \frac{y_{max}-y_{min}}{H}$$

Fungsi di bawah sama dengan `voc_to_yolo.py` di repo, ditambah dua pengaman: koordinat dipotong supaya tidak keluar dari gambar, dan kotak berukuran nol dibuang.

In [ ]:
def voc_to_yolo_lines(xml_path):
    _, w, h, objects = read_voc(xml_path)
    lines = []
    for name, x1, y1, x2, y2 in objects:
        if name not in CLASSES:
            continue                      # lewati 'person'
        x1, x2 = max(0, x1), min(w, x2)
        y1, y2 = max(0, y1), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue                      # kotak rusak
        xc, yc = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
        bw, bh = (x2 - x1) / w, (y2 - y1) / h
        lines.append(f"{CLASSES[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    return lines

print(read_voc(xml_files[0])[3][:2])
print(voc_to_yolo_lines(xml_files[0])[:2])

**Cek pemahaman:** objek pertama punya `xmin=357, xmax=404` di gambar 416 px.
Hitung manual `x_tengah = (357+404)/2/416`. Hasilnya harus sama dengan angka kedua di baris YOLO.

## Tahap 5 — Membagi data train / validation

- **train**: dipakai model untuk belajar.
- **val**: tidak pernah dilihat saat belajar, dipakai untuk mengukur apakah model benar-benar paham atau cuma menghafal.

Pembagian diacak dengan `SEED` tetap, supaya hasilnya bisa diulang. Struktur folder yang dibutuhkan YOLO:

```
dataset/
├── images/train   images/val
└── labels/train   labels/val
```

In [ ]:
items = xml_files[:]
random.Random(SEED).shuffle(items)
if MAX_IMAGES:
    items = items[:MAX_IMAGES]
n_val = int(len(items) * VAL_RATIO)
splits = {"val": items[:n_val], "train": items[n_val:]}

shutil.rmtree(DATASET, ignore_errors=True)
for split, xs in splits.items():
    (DATASET / "images" / split).mkdir(parents=True)
    (DATASET / "labels" / split).mkdir(parents=True)
    for x in xs:
        fname = read_voc(x)[0]
        shutil.copy(IMG_SRC / fname, DATASET / "images" / split / fname)
        (DATASET / "labels" / split / (Path(fname).stem + ".txt")).write_text("\n".join(voc_to_yolo_lines(x)))
    print(f"{split:5s}: {len(xs)} gambar")

File konfigurasi dataset memberi tahu YOLO letak folder dan nama kelas (sama seperti `helmet.yaml` di repo).

In [ ]:
DATA_YAML = WORK / "helmet_data.yaml"
DATA_YAML.write_text(
    f"path: {DATASET.as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    f"names: {NAMES}\n"
)
print(DATA_YAML.read_text())

## Tahap 6 — Mengecek ulang label

Kesalahan konversi label adalah penyebab paling umum model yang "tidak belajar apa-apa".
Kita gambar ulang label YOLO di atas gambar dengan rumus kebalikan (denormalisasi). Kotak harus pas di kepala.

In [ ]:
def show_yolo(img_path):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    d = ImageDraw.Draw(img)
    lbl = DATASET / "labels" / img_path.parent.name / (img_path.stem + ".txt")
    for line in lbl.read_text().splitlines():
        c, xc, yc, bw, bh = line.split()
        c, xc, yc, bw, bh = int(c), float(xc) * W, float(yc) * H, float(bw) * W, float(bh) * H
        d.rectangle([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], outline=["lime", "red"][c], width=2)
    return img

train_imgs = sorted((DATASET / "images" / "train").iterdir())
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, p in zip(axes, random.Random(2).sample(train_imgs, 3)):
    ax.imshow(show_yolo(p)); ax.set_title(p.name); ax.axis("off")
plt.show()

## Tahap 7 — Training

Parameter penting `model.train()`:

| parameter | arti |
|---|---|
| `data` | file yaml dataset |
| `epochs` | jumlah putaran belajar |
| `imgsz` | ukuran gambar input |
| `batch` | gambar per langkah |
| `patience` | berhenti lebih awal kalau mAP val tidak membaik selama N epoch (mencegah overfitting) |
| `project`, `name` | folder hasil: `runs/helmet` |

Selama training, YOLO otomatis melakukan **augmentasi** (mosaic, flip, perubahan warna/HSV, skala) supaya model tahan terhadap variasi pencahayaan dan sudut kamera di pabrik.

Di Colab T4, 30 epoch × 5000 gambar butuh sekitar 30–45 menit.

In [ ]:
model = YOLO(BASE_MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=10,
    seed=SEED,
    project=str(RUNS),   # path absolut -> hasil tepat di runs/helmet
    name="helmet",
    exist_ok=True,
)
RUN = Path(model.trainer.save_dir)
print("Hasil training ada di:", RUN)
print(sorted(p.name for p in RUN.iterdir()))

## Tahap 8 — Membaca kurva training

`results.csv` berisi nilai per epoch. Ada tiga jenis *loss* (makin kecil makin baik):

- **box_loss**: seberapa meleset posisi/ukuran kotak.
- **cls_loss**: seberapa salah menebak kelas (helm vs tanpa helm).
- **dfl_loss**: *distribution focal loss*, ketajaman tepi kotak.

Cara membaca:
- train loss dan val loss sama-sama turun → model sedang belajar dengan sehat.
- train loss terus turun tapi val loss naik → **overfitting** (model menghafal data train).
- semua loss datar sejak awal → cek label (Tahap 6) atau learning rate.

In [ ]:
df = pd.read_csv(RUN / "results.csv")
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, name in zip(axes[:3], ["box_loss", "cls_loss", "dfl_loss"]):
    ax.plot(df["epoch"], df[f"train/{name}"], label="train")
    ax.plot(df["epoch"], df[f"val/{name}"], label="val")
    ax.set_title(name); ax.set_xlabel("epoch"); ax.legend()
for col in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
    axes[3].plot(df["epoch"], df[col], label=col.split("/")[1])
axes[3].set_title("metrik validasi"); axes[3].set_xlabel("epoch"); axes[3].legend()
plt.tight_layout(); plt.show()
df.tail()

## Tahap 9 — Evaluasi

Kita evaluasi bobot terbaik (`best.pt`) pada data val.

| metrik | pertanyaan yang dijawab |
|---|---|
| **Precision** | Dari semua yang ditandai "tanpa helm", berapa persen yang benar? (rendah = banyak alarm palsu) |
| **Recall** | Dari semua orang yang benar-benar tanpa helm, berapa persen yang tertangkap? (rendah = pelanggaran lolos) |
| **mAP50** | Rata-rata ketepatan deteksi kalau kotak cukup tumpang-tindih 50% dengan kotak asli (IoU ≥ 0.5) |
| **mAP50-95** | Sama, tapi dirata-rata untuk IoU 0.5 sampai 0.95 (lebih ketat soal posisi kotak) |

Untuk keselamatan kerja, **recall `no_helmet`** biasanya paling penting: lebih baik ada sedikit alarm palsu daripada pelanggaran yang lolos.

In [ ]:
best = YOLO(RUN / "weights" / "best.pt")
# mAP dihitung dengan ambang sangat rendah (conf=0.001, standar Ultralytics) supaya seluruh kurva precision-recall terhitung
m = best.val(data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)

per_class = []
for i, c in enumerate(m.box.ap_class_index):
    per_class.append({
        "name": NAMES[int(c)],
        "precision": float(m.box.p[i]), "recall": float(m.box.r[i]),
        "map50": float(m.box.ap50[i]), "map50_95": float(m.box.ap[i]),
    })
overall = {"precision": float(m.box.mp), "recall": float(m.box.mr), "map50": float(m.box.map50), "map50_95": float(m.box.map)}
pd.DataFrame(per_class).set_index("name").round(3)

### Confusion matrix sebagai heatmap

Baris = **prediksi** model, kolom = **label asli**. Ada kelas tambahan `background`:

- kolom `background` = model melihat kepala padahal tidak ada (*false positive*).
- baris `background` = kepala asli yang tidak terdeteksi sama sekali (*false negative*).

Kita normalisasi per kolom, sehingga diagonal = persentase objek asli yang terdeteksi dengan kelas yang benar.

**Penting:** mAP di atas dihitung dengan ambang keyakinan hampir nol (0.001) supaya seluruh kurva precision-recall terhitung. Kalau confusion matrix memakai ambang yang sama, ribuan tebakan ber-skor rendah ikut dihitung sebagai alarm palsu. Karena itu confusion matrix dihitung ulang pada ambang kerja `CONF_CM = 0.4`, yaitu ambang yang benar-benar dipakai di lapangan.

In [ ]:
# Confusion matrix dihitung pada ambang kerja (sama dengan default website).
# plots=True wajib: Ultralytics hanya mengisi confusion matrix kalau plot dibuat (disimpan di runs/helmet/val)
CONF_CM = 0.4
cmv = best.val(data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, conf=CONF_CM, plots=True, verbose=False,
               project=str(RUN), name="val", exist_ok=True)
cm = cmv.confusion_matrix.matrix        # shape (nc+1, nc+1): [prediksi, asli]
labels = NAMES + ["background"]
cm_norm = cm / np.maximum(cm.sum(0, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)), labels); ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel("label asli"); ax.set_ylabel("prediksi")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color="white" if cm_norm[i, j] > 0.5 else "black")
fig.colorbar(im); plt.title("Confusion matrix (dinormalisasi)"); plt.show()

### Heatmap posisi kepala di data train

Heatmap kedua menunjukkan **di mana** kepala biasanya berada di dalam gambar.
Kalau semua kepala berkumpul di tengah, model bisa kesulitan menangkap orang di pinggir frame CCTV. Ini bias dataset yang perlu diketahui sebelum dipasang di pabrik.

In [ ]:
BINS = 20
heat = np.zeros((BINS, BINS))
for lbl in (DATASET / "labels" / "train").glob("*.txt"):
    for line in lbl.read_text().splitlines():
        _, xc, yc, *_ = map(float, line.split())
        heat[min(int(yc * BINS), BINS - 1), min(int(xc * BINS), BINS - 1)] += 1

plt.figure(figsize=(5, 4))
plt.imshow(heat, cmap="inferno", extent=[0, 1, 1, 0])
plt.colorbar(label="jumlah kepala"); plt.xlabel("x (kiri → kanan)"); plt.ylabel("y (atas → bawah)")
plt.title("Sebaran posisi kepala (train)"); plt.show()

Simpan semua angka evaluasi ke `metrics.json` di folder hasil training. Website membaca file ini untuk menggambar heatmap interaktif.

In [ ]:
metrics = {
    "names": NAMES,
    "overall": overall,
    "per_class": per_class,
    "confusion": {"labels": labels, "matrix": cm.astype(int).tolist()},
    "label_heatmap": heat.astype(int).tolist(),
    "counts": {k: len(v) for k, v in splits.items()},
    "config": {"base_model": BASE_MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "conf_cm": CONF_CM},
}
(RUN / "metrics.json").write_text(json.dumps(metrics, indent=1))
print("Tersimpan:", RUN / "metrics.json")

## Tahap 10 — Inference: menghitung orang tanpa helm

`conf` adalah ambang keyakinan. Deteksi dengan skor di bawahnya dibuang.

- `conf` rendah (0.25): lebih banyak pelanggaran tertangkap, tapi alarm palsu juga lebih banyak.
- `conf` tinggi (0.6): lebih sedikit alarm palsu, tapi ada pelanggaran yang lolos.

Coba ubah `CONF` dan bandingkan.

In [ ]:
CONF = 0.4
val_imgs = sorted((DATASET / "images" / "val").iterdir())
samples = random.Random(3).sample(val_imgs, 4)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, p in zip(axes, samples):
    r = best.predict(p, conf=CONF, verbose=False)[0]
    n_no = int((r.boxes.cls == 1).sum())
    ax.imshow(r.plot()[:, :, ::-1])     # plot() menghasilkan BGR -> balik ke RGB
    ax.set_title(f"{n_no} orang tanpa helm" if n_no else "semua pakai helm", color="red" if n_no else "green")
    ax.axis("off")
plt.show()

**Latihan:** jalankan sel di atas dengan `CONF = 0.25` lalu `CONF = 0.6`. Di gambar mana hasilnya berubah?

## Tahap 11 — Menyimpan hasil untuk website

Website membaca folder `runs/helmet` (berisi `weights/best.pt`, `results.csv`, `metrics.json`, dan gambar-gambar plot).

1. Jalankan sel di bawah untuk membuat `helmet_run.zip`. Di Colab, file akan otomatis terunduh.
2. Ekstrak isinya ke folder project website sehingga ada `runs/helmet/weights/best.pt`.
3. Jalankan `python app.py` lalu buka http://127.0.0.1:5000.

In [ ]:
zip_path = shutil.make_archive(str(WORK / "helmet_run"), "zip", root_dir=RUN.parent, base_dir=RUN.name)
print("Zip:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass  # bukan di Colab: file zip ada di folder kerja

## Langkah berikutnya

- **Model lebih besar:** ganti `BASE_MODEL = "yolov8s.pt"`. Biasanya mAP naik beberapa poin, inference sedikit lebih lambat.
- **Data dari pabrikmu sendiri:** foto dari kamera CCTV pabrik, beri label (misalnya dengan Roboflow atau Label Studio), lalu gabungkan dengan dataset ini. Ini cara paling efektif menaikkan akurasi di lokasi nyata.
- **Kelas tambahan:** rompi (*vest*), sepatu safety, masker. Tambahkan nama kelas di `NAMES` dan labelnya.
- **Real-time:** `webcam_detect.py` di repo menjalankan model pada webcam. Untuk CCTV, ganti `cv2.VideoCapture(0)` dengan alamat RTSP kamera.
- **Export:** `best.export(format="onnx")` untuk dipasang di perangkat tanpa PyTorch.